# Lab | Classification

This notebook compares multiple classification algorithms using the Palmer Penguins dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_curve, auc,
    ConfusionMatrixDisplay, classification_report
)

sns.set_style("whitegrid")

## Task 1: Data Prep & Baseline

In [ ]:
# 1. Load the Palmer Penguins dataset
penguins = sns.load_dataset("penguins").dropna()

# 2. Explore briefly
print(f"Dataset shape: {penguins.shape}")
print(f"\nSpecies distribution:\n{penguins['species'].value_counts()}")
print(f"\nColumns information:\n{penguins.info()}")
penguins.head()

### Observations:
- The dataset contains 333 samples (after dropping NaNs) and 7 columns.
- Target: `species` (Adelie, Chinstrap, Gentoo).
- Features: `island`, `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`, `sex`.
- Categorical features: `island`, `sex`.
- Numerical features: `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`.

In [ ]:
# 3. Prepare features
le = LabelEncoder()
y = le.fit_transform(penguins['species'])

X = pd.get_dummies(penguins.drop('species', axis=1), drop_first=True)

# 4. Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 5. Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# 6. Fit a LogisticRegression model
lr = LogisticRegression(max_iter=10000, multi_class='multinomial')
lr.fit(X_train_scaled, y_train)

# 7. Report metrics
y_pred_lr = lr.predict(X_test_scaled)
print("Logistic Regression Baseline Results:")
print(classification_report(y_test, y_pred_lr, target_names=le.classes_))

### Interpretation of Baseline Results:
- **Easiest to classify**: Gentoo usually shows high scores because they are physically distinct (larger) from the other two.
- **Hardest to classify**: Adelie and Chinstrap are often harder to distinguish as they share more similar physical characteristics.
- Overall, Logistic Regression performs exceptionally well, likely reaching near 100% accuracy on this relatively simple dataset.

## Task 2: Algorithm Comparison

In [ ]:
models = {
    "GaussianNB": GaussianNB(),
    "SVC Linear": SVC(kernel="linear", probability=True),
    "SVC RBF": SVC(kernel="rbf", probability=True),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "RandomForest": RandomForestClassifier(random_state=42)
}

results = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average='weighted'),
        "Recall": recall_score(y_test, y_pred, average='weighted'),
        "F1": f1_score(y_test, y_pred, average='weighted')
    })

comparison_df = pd.DataFrame(results).sort_values(by="F1", ascending=False)
comparison_df

### Discussion:
- Many models perform perfectly or near-perfectly on this test set.
- **SVC Linear** and **RandomForest** often hit 100% accuracy.
- This suggests the classes are well-separated in the feature space once scaled and encoded.
- GaussianNB might lag slightly if the feature distributions aren't strictly Gaussian, but usually still performs well.

## Task 3: Confusion Matrices & ROC Curves

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

all_models = {"LogisticRegression": lr, **models}

for i, (name, model) in enumerate(all_models.items()):
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
    disp.plot(ax=axes[i], cmap='Blues', colorbar=False)
    axes[i].set_title(name)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import label_binarize
from itertools import cycle

y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
n_classes = y_test_bin.shape[1]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

colors = ['blue', 'red', 'green']

for idx, (name, model) in enumerate(all_models.items()):
    y_score = model.predict_proba(X_test_scaled)
    
    for i, color in zip(range(n_classes), colors):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
        roc_auc = auc(fpr, tpr)
        axes[idx].plot(fpr, tpr, color=color, lw=2,
                 label=f'{le.classes_[i]} (AUC = {roc_auc:.2f})')

    axes[idx].plot([0, 1], [0, 1], 'k--', lw=2)
    axes[idx].set_xlim([0.0, 1.0])
    axes[idx].set_ylim([0.0, 1.05])
    axes[idx].set_xlabel('False Positive Rate')
    axes[idx].set_ylabel('True Positive Rate')
    axes[idx].set_title(f'ROC Curve: {name}')
    axes[idx].legend(loc="lower right")

plt.tight_layout()
plt.show()

### Discussion:
- **Balance**: Random Forest and SVC Linear show excellent balance, with AUC values often at 1.00.
- **Hardest Pair**: Adelie and Chinstrap are the most likely to be confused if any errors occur, as they are similar in size and live in similar habitats.
- **Recommendation**: Random Forest is highly recommended for its robustness and naturally high performance on this dataset without much tuning.

## Task 4: Hyperparameter Exploration

In [ ]:
# Selecting RandomForest as the best model to tune
rf = RandomForestClassifier(random_state=42)

param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10]
}

grid_search = GridSearchCV(rf, param_grid, cv=5, scoring="f1_weighted", n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV F1 Score: {grid_search.best_score_:.4f}")

best_rf = grid_search.best_estimator_
y_pred_tuned = best_rf.predict(X_test_scaled)

print("\nTuned Random Forest Results:")
print(classification_report(y_test, y_pred_tuned, target_names=le.classes_))

### Reflection:
- **Improvement**: If the initial performance was already 100%, improvement might not be visible numerically, but tuning helps in making the model more robust.
- **Overfitting**: 5-fold CV helps mitigate overfitting to specific splits.
- **Impact**: Hyperparameter tuning is most impactful when the dataset is noisy, imbalanced, or when the default parameters lead to underfitting/overfitting.